# 09 – Train SWIN LSTM On Sequences (NTHU)

Tujuan Notebook:
Melatih dan membandingkan 2 konfigurasi temporal model di atas fitur SWIN:

| ID | Backbone | Temporal Model |
|----|----------|----------------|
| 1  | SWIN     | LSTM (2-layer, Stacked) |
| 2  | SWIN     | BiLSTM (2-layer) |

Setiap model dilatih dengan Attention Mechanism (Additive/Bahdanau-style) yang bisa di-toggle.

Input: sequences_nthuddd2/SWIN/SWIN_Seq_{Split}_NTHUD.pt  
Output: lstm_models_nthuddd2/SWIN_{LSTM_TYPE}/SWIN_{LSTM_TYPE}_BEST.pth, _LAST.pth, _history.json

Catatan penting:
- BiLSTM hanya valid untuk evaluasi offline dan analisis skripsi.
- LSTM standar adalah kandidat utama untuk deployment real-time.
- Notebook ini khusus SWIN agar evaluasi lebih fokus dan praktis.

# CELL 1 — Import & Konfigurasi Global

In [16]:
import os, gc, json, time, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts, ReduceLROnPlateau
from sklearn.metrics import (
    f1_score, accuracy_score, precision_score,
    recall_score, confusion_matrix, roc_auc_score
)
from collections import Counter
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

# ─── SEED ──────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = True

# ─── DEVICE ────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ─── PATH ──────────────────────────────────────────────────
BASE_DIR      = r"C:\kuliah-sementara\SKRIPSI"
SEQ_DIR       = os.path.join(BASE_DIR, "sequences_nthuddd2")
SAVE_DIR      = os.path.join(BASE_DIR, "lstm_models_nthuddd2")
os.makedirs(SAVE_DIR, exist_ok=True)

# ─── DATA CONFIG (SWIN ONLY) ───────────────────────────────
MODELS  = ["SWIN"]
SPLITS  = ["train", "val", "test"]
INPUT_DIM   = 512
WINDOW_SIZE = 30
LABEL_MAP   = {0: "drowsy", 1: "notdrowsy"}
NUM_CLASSES = 2

# ─── ARSITEKTUR LSTM ───────────────────────────────────────
LSTM_HIDDEN_DIM  = 256
LSTM_NUM_LAYERS  = 2
LSTM_DROPOUT     = 0.3
FC_DROPOUT       = 0.4
FC_ACTIVATION    = "gelu"
USE_ATTENTION    = True

# ─── TRAINING HYPERPARAMETER ───────────────────────────────
BATCH_SIZE       = 128
MAX_EPOCHS       = 80
LEARNING_RATE    = 2e-4
WEIGHT_DECAY     = 5e-4
PATIENCE         = 15
MAX_GRAD_NORM    = 5.0

# ─── SCHEDULER ─────────────────────────────────────────────
SCHEDULER_TYPE   = "cosine_warm"
T_0              = 10
T_MULT           = 2
ETA_MIN          = 1e-6

# ─── LOSS ──────────────────────────────────────────────────
LOSS_TYPE        = "ce"
LABEL_SMOOTHING  = 0.05

# ─── AUGMENTASI (EXP_D Configuration) ──────────────────────
USE_AUG          = True
AUG_PROB_FD      = 0.3
AUG_PROB_GN      = 0.4
N_DROP_MAX       = 1
NOISE_STD        = 0.01

# ─── OPTIMIZER ─────────────────────────────────────────────
OPTIMIZER_TYPE   = "adamw"

# ─── DATALOADER ────────────────────────────────────────────
NUM_WORKERS      = 0

# ─── PRINT KONFIGURASI ─────────────────────────────────────
print("=" * 75)
print("  09 — TRAIN SWIN_LSTM_EXP_D (Safety-Optimized Configuration)")
print("=" * 75)
print(f"  Device          : {DEVICE}")
if DEVICE.type == "cuda":
    print(f"  GPU             : {torch.cuda.get_device_name(0)}")
    print(f"  VRAM total      : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"  Sequences dir   : {SEQ_DIR}")
print(f"  Save dir        : {SAVE_DIR}")
print()
print("  ─── DATA ───")
print("  Backbone       : SWIN only")
print(f"  Input dim       : {INPUT_DIM}")
print(f"  Window size     : {WINDOW_SIZE}")
print()
print("  ─── ARSITEKTUR ───")
print(f"  LSTM hidden     : {LSTM_HIDDEN_DIM}")
print(f"  LSTM layers     : {LSTM_NUM_LAYERS}")
print(f"  FC Activation   : {FC_ACTIVATION}")
print(f"  Attention       : {USE_ATTENTION}")
print()
print("  ─── TRAINING ───")
print(f"  Batch size      : {BATCH_SIZE}")
print(f"  Max epochs      : {MAX_EPOCHS}")
print(f"  Early stop pat. : {PATIENCE}")
print(f"  Learning rate   : {LEARNING_RATE}")
print(f"  Weight decay    : {WEIGHT_DECAY}")
print(f"  Max grad norm   : {MAX_GRAD_NORM}")
print(f"  Optimizer       : {OPTIMIZER_TYPE}")
print(f"  Label Smoothing : {LABEL_SMOOTHING}")
print()
print("  ─── SCHEDULER ───")
print(f"  Type            : {SCHEDULER_TYPE}")
print(f"  T_0             : {T_0} epochs")
print(f"  T_mult          : {T_MULT}x")
print(f"  Eta min         : {ETA_MIN}")
print()
print("  ─── AUGMENTATION (EXP_D) ───")
print(f"  Enabled         : {USE_AUG}")
print(f"  Frame drop prob : {AUG_PROB_FD}")
print(f"  Noise prob      : {AUG_PROB_GN}")
print(f"  Max drop frames : {N_DROP_MAX}")
print(f"  Noise std       : {NOISE_STD}")
print("=" * 75)


  09 — TRAIN SWIN_LSTM_EXP_D (Safety-Optimized Configuration)
  Device          : cuda
  GPU             : NVIDIA GeForce RTX 3060
  VRAM total      : 12.9 GB
  Sequences dir   : C:\kuliah-sementara\SKRIPSI\sequences_nthuddd2
  Save dir        : C:\kuliah-sementara\SKRIPSI\lstm_models_nthuddd2

  ─── DATA ───
  Backbone       : SWIN only
  Input dim       : 512
  Window size     : 30

  ─── ARSITEKTUR ───
  LSTM hidden     : 256
  LSTM layers     : 2
  FC Activation   : gelu
  Attention       : True

  ─── TRAINING ───
  Batch size      : 128
  Max epochs      : 80
  Early stop pat. : 15
  Learning rate   : 0.0002
  Weight decay    : 0.0005
  Max grad norm   : 5.0
  Optimizer       : adamw
  Label Smoothing : 0.05

  ─── SCHEDULER ───
  Type            : cosine_warm
  T_0             : 10 epochs
  T_mult          : 2x
  Eta min         : 1e-06

  ─── AUGMENTATION (EXP_D) ───
  Enabled         : True
  Frame drop prob : 0.3
  Noise prob      : 0.4
  Max drop frames : 1
  Noise std      

## 2. Validasi File Sekuens Input (SWIN)

Pastikan 3 file `.pt` SWIN (train, val, test) tersedia sebelum training dimulai.

In [17]:
print("[VALIDASI] Mengecek file sekuens input SWIN...\n")

all_ok = True
for model in MODELS:
    for split in SPLITS:
        fname = f"{model}_Seq_{split.capitalize()}_NTHUD.pt"
        fpath = os.path.join(SEQ_DIR, model, fname)
        exists = os.path.exists(fpath)

        if exists:
            data = torch.load(fpath, map_location="cpu", weights_only=False)
            n_seq   = data["features"].shape[0]
            n_drow  = int((data["labels"] == 0).sum())
            n_notd  = int((data["labels"] == 1).sum())
            balance = n_drow / n_seq * 100
            print(f"  [OK] {fname}")
            print(f"       Shape={list(data['features'].shape)} | "
                  f"drowsy={n_drow:,} ({balance:.1f}%) | notdrowsy={n_notd:,}")
        else:
            print(f"  [MISSING] {fname}")
            all_ok = False

print()
if not all_ok:
    raise FileNotFoundError(
        "File SWIN .pt belum lengkap!\n"
        "Pastikan notebook 08_Build_Sequences_30Frames_NTHU sudah dijalankan untuk SWIN."
    )
print("✓ Semua file sekuens SWIN tersedia. Siap melatih SWIN-LSTM.")

[VALIDASI] Mengecek file sekuens input SWIN...

  [OK] SWIN_Seq_Train_NTHUD.pt
       Shape=[8053, 30, 512] | drowsy=4,502 (55.9%) | notdrowsy=3,551
  [OK] SWIN_Seq_Val_NTHUD.pt
       Shape=[3634, 30, 512] | drowsy=2,100 (57.8%) | notdrowsy=1,534
  [OK] SWIN_Seq_Test_NTHUD.pt
       Shape=[1302, 30, 512] | drowsy=548 (42.1%) | notdrowsy=754

✓ Semua file sekuens SWIN tersedia. Siap melatih SWIN-LSTM.


## 3. Dataset Class & DataLoader

SequenceDataset membungkus file `.pt` SWIN menjadi PyTorch Dataset standar.
Normalisasi per-feature (Z-Score) diterapkan menggunakan statistik dari training set saja
untuk mencegah data leakage ke val/test.

In [18]:
class SequenceDataset(Dataset):
    """
    Dataset wrapper untuk file sekuens .pt.
    Mendukung optional Z-Score normalisasi menggunakan
    mean/std dari training set.
    """
    def __init__(self, features: torch.Tensor, labels: torch.Tensor,
                 mean: torch.Tensor = None, std: torch.Tensor = None):
        self.features = features.float()
        self.labels   = labels.long()

        # Terapkan normalisasi jika tersedia
        if mean is not None and std is not None:
            # Clamp std agar tidak ada pembagian dengan 0
            std_safe = std.clamp(min=1e-8)
            self.features = (self.features - mean) / std_safe

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]


class SequenceDatasetAugmented(Dataset):
    """
    Dataset wrapper dengan optional Temporal Augmentation untuk training.

    Temporal Augmentation:
    1. Frame Dropping  : Hapus 1-2 frame secara acak dari 30 frame,
                         gantikan dengan copy frame tetangga terdekat.
                         Tujuan: simulasi kamera lag / dropped frame.
    2. Gaussian Noise  : Tambahkan noise N(0, 0.01) ke semua fitur.
                         Tujuan: membuat LSTM lebih robust terhadap
                         variasi kecil pada fitur CNN.
    """
    def __init__(self, features: torch.Tensor, labels: torch.Tensor,
                 mean: torch.Tensor = None, std: torch.Tensor = None,
                 augment: bool = False,
                 aug_prob_fd: float = 0.3,
                 aug_prob_gn: float = 0.3,
                 n_drop_max: int = 1,
                 noise_std: float = 0.01):
        self.features    = features.float()
        self.labels      = labels.long()
        self.augment     = augment
        self.aug_prob_fd = aug_prob_fd
        self.aug_prob_gn = aug_prob_gn
        self.n_drop_max  = n_drop_max
        self.noise_std   = noise_std

        # Z-Score Normalisasi
        if mean is not None and std is not None:
            std_safe = std.clamp(min=1e-8)
            self.features = (self.features - mean) / std_safe

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        feat = self.features[idx].clone()  # [T=30, D=512]

        if self.augment:
            # ── Augmentasi 1: Frame Dropping ──────────────────
            if random.random() < self.aug_prob_fd:
                n_drop = random.randint(1, self.n_drop_max)
                T = feat.shape[0]
                drop_idx = sorted(random.sample(range(T), n_drop), reverse=True)
                for di in drop_idx:
                    neighbor = di - 1 if di > 0 else di + 1
                    neighbor = max(0, min(neighbor, T - 1))
                    feat[di] = feat[neighbor].clone()

            # ── Augmentasi 2: Gaussian Noise ──────────────────
            if random.random() < self.aug_prob_gn:
                noise = torch.randn_like(feat) * self.noise_std
                feat = feat + noise

        return feat, self.labels[idx]


def load_split_data(model_name: str, split: str):
    """Load file .pt untuk satu backbone + satu split."""
    fname = f"{model_name}_Seq_{split.capitalize()}_NTHUD.pt"
    fpath = os.path.join(SEQ_DIR, model_name, fname)
    data  = torch.load(fpath, map_location="cpu", weights_only=False)
    return data["features"].float(), data["labels"].long()


def compute_train_stats(features: torch.Tensor):
    """
    Hitung mean & std dari training set untuk Z-Score normalisasi.
    Shape features: [N_seq, T, D] → statistik dihitung per dimensi D.
    """
    N, T, D = features.shape
    flat    = features.reshape(-1, D)   # [N*T, D]
    mean    = flat.mean(dim=0)          # [D]
    std     = flat.std(dim=0)           # [D]
    return mean, std


def build_dataloaders(model_name: str, batch_size: int = 64, use_aug: bool = False):
    """
    Build train/val/test DataLoader untuk satu backbone.
    Normalisasi dihitung dari train set lalu diterapkan ke semua split.

    Returns:
        loaders    : dict {"train": ..., "val": ..., "test": ...}
        class_weight : Tensor untuk weighted CrossEntropyLoss
        norm_stats : dict {"mean": ..., "std": ...}
    """
    # Load semua split
    train_feat, train_lab = load_split_data(model_name, "train")
    val_feat,   val_lab   = load_split_data(model_name, "val")
    test_feat,  test_lab  = load_split_data(model_name, "test")

    # Hitung normalisasi dari train set saja
    mean, std = compute_train_stats(train_feat)

    # Buat dataset dengan augmentation toggle
    if use_aug:
        train_ds = SequenceDatasetAugmented(
            train_feat, train_lab, mean, std,
            augment=True,
            aug_prob_fd=AUG_PROB_FD,
            aug_prob_gn=AUG_PROB_GN,
            n_drop_max=N_DROP_MAX,
            noise_std=NOISE_STD,
        )
    else:
        train_ds = SequenceDataset(train_feat, train_lab, mean, std)

    val_ds  = SequenceDataset(val_feat,   val_lab,   mean, std)
    test_ds = SequenceDataset(test_feat,  test_lab,  mean, std)

    # ── Class Weight untuk imbalance handling ─────────────────
    label_counts = Counter(train_lab.numpy())
    n_total      = len(train_lab)
    weights      = torch.tensor([
        n_total / (NUM_CLASSES * label_counts[c])
        for c in range(NUM_CLASSES)
    ], dtype=torch.float32)

    print(f"  [Normalisasi] feat_mean={mean.mean():.4f}, feat_std={std.mean():.4f}")
    print(f"  [ClassWeight] drowsy={weights[0]:.4f}, notdrowsy={weights[1]:.4f}")
    print(f"  [Augmentation] {'ENABLED' if use_aug else 'disabled'}")

    loaders = {
        "train": DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                            num_workers=NUM_WORKERS, pin_memory=(DEVICE.type=="cuda")),
        "val"  : DataLoader(val_ds,   batch_size=batch_size, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=(DEVICE.type=="cuda")),
        "test" : DataLoader(test_ds,  batch_size=batch_size, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=(DEVICE.type=="cuda")),
    }

    norm_stats = {"mean": mean, "std": std}
    return loaders, weights, norm_stats


print("✓ Fungsi Dataset, SequenceDatasetAugmented, dan DataLoader terdefinisi.")


✓ Fungsi Dataset, SequenceDatasetAugmented, dan DataLoader terdefinisi.


## 4. Arsitektur Model

### Additive Attention (Bahdanau-style)
Mekanisme Attention bekerja seperti "penilai kepentingan" — dari 30 output LSTM (satu per frame),
model belajar frame mana yang paling informatif untuk keputusan akhir.
Analogi: dari rekaman 1 detik, model otomatis fokus ke momen ketika mata mulai menutup,
bukan rata-rata semua 30 frame.

### LSTM vs BiLSTM
- **LSTM**: membaca sequence dari frame 1 → 30. Cocok real-time.
- **BiLSTM**: membaca frame 1→30 DAN 30→1, lalu menggabungkan hasilnya.
  Lebih akurat untuk analisis offline karena melihat "konteks penuh."

In [19]:
class AdditiveAttention(nn.Module):
    """
    Additive (Bahdanau-style) Attention untuk output LSTM.

    Input  : lstm_out → [B, T, H]  (B = Batch Size, T = Time Steps, H = Hidden Dimension)
    Output : context → [B, H]      (weighted sum dari semua timestep)

    Cara kerja:
      1. Setiap timestep t mendapat skor energi e_t = v^T tanh(W * h_t)
      2. Skor di-softmax → attention weights α_t
      3. Context vector = Σ(α_t * h_t)
    """
    def __init__(self, hidden_dim: int):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, hidden_dim)
        self.v    = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, lstm_out: torch.Tensor):
        # lstm_out: [B, T, H]
        energy   = torch.tanh(self.attn(lstm_out))     # [B, T, H]
        scores   = self.v(energy).squeeze(-1)           # [B, T]
        weights  = torch.softmax(scores, dim=-1)        # [B, T]
        context  = torch.bmm(
            weights.unsqueeze(1), lstm_out
        ).squeeze(1)                                    # [B, H]
        return context, weights


# ── Activation Factory ────────────────────────────────────────
_ACTIVATION_MAP = {
    "relu": nn.ReLU,
    "gelu": nn.GELU,
    "silu": nn.SiLU,
    "elu":  nn.ELU,
}

def get_activation(name: str) -> nn.Module:
    """Get activation function by name."""
    name = name.lower()
    if name not in _ACTIVATION_MAP:
        raise ValueError(f"Aktivasi '{name}' tidak dikenal. Pilih: {list(_ACTIVATION_MAP.keys())}")
    return _ACTIVATION_MAP[name]()


class DrowsinessLSTM(nn.Module):
    """
    Arsitektur utama: Stacked LSTM/BiLSTM + optional Attention + FC Classifier.

    Args:
        input_dim       : dimensi fitur per frame (512)
        hidden_dim      : unit hidden LSTM (256)
        num_layers      : jumlah layer LSTM (2)
        num_classes     : jumlah kelas (2)
        bidirectional   : True = BiLSTM, False = LSTM standar
        use_attention   : True = tambahkan Additive Attention
        lstm_dropout    : dropout antar layer LSTM (hanya berlaku jika num_layers > 1)
        fc_dropout      : dropout sebelum FC classifier
        fc_activation   : "relu", "gelu", "silu" — GELU untuk SWIN
    """
    def __init__(
        self,
        input_dim       : int   = 512,
        hidden_dim      : int   = 256,
        num_layers      : int   = 2,
        num_classes     : int   = 2,
        bidirectional   : bool  = False,
        use_attention   : bool  = True,
        lstm_dropout    : float = 0.3,
        fc_dropout      : float = 0.5,
        fc_activation   : str   = "relu",
    ):
        super().__init__()
        self.bidirectional = bidirectional
        self.use_attention = use_attention
        self.hidden_dim    = hidden_dim
        self.num_layers    = num_layers
        num_directions     = 2 if bidirectional else 1
        lstm_out_dim       = hidden_dim * num_directions  # 256 atau 512

        # ── Layer Normalisasi Input ──────────────────────────
        self.input_norm = nn.LayerNorm(input_dim)

        # ── LSTM / BiLSTM ────────────────────────────────────
        self.lstm = nn.LSTM(
            input_size    = input_dim,
            hidden_size   = hidden_dim,
            num_layers    = num_layers,
            batch_first   = True,
            bidirectional = bidirectional,
            dropout       = lstm_dropout if num_layers > 1 else 0.0,
        )

        # ── Attention ────────────────────────────────────────
        if use_attention:
            self.attention = AdditiveAttention(lstm_out_dim)
            classifier_in  = lstm_out_dim
        else:
            self.attention  = None
            classifier_in   = lstm_out_dim

        # ── FC Classifier dengan flexible activation ────────
        act = get_activation(fc_activation)
        self.classifier = nn.Sequential(
            nn.Dropout(fc_dropout),
            nn.Linear(classifier_in, 128),
            act,  # GELU untuk SWIN, ReLU untuk others
            nn.Dropout(fc_dropout * 0.5),
            nn.Linear(128, num_classes),
        )

    def forward(self, x: torch.Tensor):
        # x: [B, T, input_dim]
        x = self.input_norm(x)                          # [B, T, D]
        lstm_out, (h_n, _) = self.lstm(x)               # [B, T, H*num_dir]

        if self.use_attention:
            context, attn_weights = self.attention(lstm_out)  # [B, H*num_dir]
        else:
            context      = lstm_out[:, -1, :]
            attn_weights = None

        logits = self.classifier(context)                # [B, num_classes]
        return logits, attn_weights


def build_model(bidirectional: bool = False, use_attention: bool = True,
                fc_activation: str = "relu") -> DrowsinessLSTM:
    """Factory function untuk membangun model dengan konfigurasi global."""
    return DrowsinessLSTM(
        input_dim       = INPUT_DIM,
        hidden_dim      = LSTM_HIDDEN_DIM,
        num_layers      = LSTM_NUM_LAYERS,
        num_classes     = NUM_CLASSES,
        bidirectional   = bidirectional,
        use_attention   = use_attention,
        lstm_dropout    = LSTM_DROPOUT,
        fc_dropout      = FC_DROPOUT,
        fc_activation   = fc_activation,
    ).to(DEVICE)


# ── Tes arsitektur dengan dummy input ──────────────────────
print("[TEST] Arsitektur LSTM + GELU (SWIN config):")
_m = build_model(bidirectional=False, fc_activation="gelu")
_x = torch.randn(4, WINDOW_SIZE, INPUT_DIM).to(DEVICE)
_out, _attn = _m(_x)
print(f"  Input  : {list(_x.shape)}")
print(f"  Output : {list(_out.shape)}")
print(f"  Attn   : {list(_attn.shape) if _attn is not None else 'None'}")
total_params = sum(p.numel() for p in _m.parameters() if p.requires_grad)
print(f"  Params : {total_params:,}")
del _m, _x, _out, _attn
torch.cuda.empty_cache()

print("\n✓ Arsitektur LSTM dengan flexible activation terdefinisi.")


[TEST] Arsitektur LSTM + GELU (SWIN config):
  Input  : [4, 30, 512]
  Output : [4, 2]
  Attn   : [4, 30]
  Params : 1,415,042

✓ Arsitektur LSTM dengan flexible activation terdefinisi.


## 5. Fungsi Training Core

Satu epoch training + satu epoch evaluasi (val/test).
Fungsi ini dipakai ulang untuk semua kombinasi model.

In [20]:
def train_epoch(model, loader, criterion, optimizer, max_grad_norm=None):
    """Jalankan satu epoch training. Returns loss, accuracy, f1_macro."""
    if max_grad_norm is None:
        max_grad_norm = MAX_GRAD_NORM
    
    model.train()
    total_loss = 0.0
    all_preds, all_labels = [], []

    for feats, labels in loader:
        feats  = feats.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()
        logits, _ = model(feats)
        loss      = criterion(logits, labels)
        loss.backward()

        # Gradient clipping — cegah exploding gradients pada LSTM
        # SWIN uses max_grad_norm=5.0 (lebih fleksibel)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=max_grad_norm)

        optimizer.step()

        total_loss += loss.item() * len(labels)
        preds = logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    acc      = accuracy_score(all_labels, all_preds)
    f1       = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    return avg_loss, acc, f1


@torch.no_grad()
def evaluate(model, loader, criterion):
    """Evaluasi model (val atau test). Returns loss, accuracy, f1_macro, preds, labels."""
    model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []

    for feats, labels in loader:
        feats  = feats.to(DEVICE)
        labels = labels.to(DEVICE)

        logits, _ = model(feats)
        loss      = criterion(logits, labels)

        total_loss += loss.item() * len(labels)
        preds = logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    acc      = accuracy_score(all_labels, all_preds)
    f1       = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    return avg_loss, acc, f1, np.array(all_preds), np.array(all_labels)


print("✓ Fungsi train_epoch dan evaluate terdefinisi.")


✓ Fungsi train_epoch dan evaluate terdefinisi.


## 6. Training Pipeline Utama

Fungsi `run_training()` adalah orkestrator lengkap yang menggabungkan:
- Build model + optimizer + scheduler
- Resume dari checkpoint (State Saving & Resume Training)
- Loop training dengan Early Stopping berbasis Val F1 Macro
- Simpan model terbaik (`_BEST.pth`) dan model terakhir (`_LAST.pth`)
- Catat semua history ke `_history.json`

In [21]:
def run_training(
    backbone_name : str,
    bidirectional : bool = False,
    use_attention : bool = True,
    force_retrain : bool = False,
) -> dict:
    lstm_type  = "BILSTM" if bidirectional else "LSTM"
    model_id   = f"{backbone_name}_{lstm_type}"
    model_dir  = os.path.join(SAVE_DIR, model_id)
    os.makedirs(model_dir, exist_ok=True)

    path_best  = os.path.join(model_dir, f"{model_id}_BEST.pth")
    path_last  = os.path.join(model_dir, f"{model_id}_LAST.pth")
    path_hist  = os.path.join(model_dir, f"{model_id}_history.json")

    print("\n" + "=" * 70)
    print(f"  MODEL: {model_id} — SWIN_EXP_D Configuration")
    print(f"  Attention: {use_attention} | Bidirectional: {bidirectional}")
    print("=" * 70)

    # ── 1. Load Data ─────────────────────────────────────────
    print("[1/7] Loading data & membangun DataLoader...")
    loaders, class_weight, norm_stats = build_dataloaders(
        backbone_name, BATCH_SIZE, use_aug=USE_AUG
    )
    class_weight = class_weight.to(DEVICE)

    # ── 2. Bangun Model ──────────────────────────────────────
    print("[2/7] Membangun model...")
    model = build_model(
        bidirectional=bidirectional,
        use_attention=use_attention,
        fc_activation=FC_ACTIVATION,  # GELU untuk SWIN
    )
    
    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Total params: {total_params:,}")

    # ── 3. Build Loss, Optimizer, Scheduler ──────────────────
    print("[3/7] Building loss, optimizer, scheduler...")
    
    # Loss dengan label smoothing (SWIN-tuned)
    criterion = nn.CrossEntropyLoss(
        weight=class_weight,
        label_smoothing=LABEL_SMOOTHING,
    )

    # Optimizer: AdamW untuk SWIN (better weight decay handling)
    if OPTIMIZER_TYPE == "adamw":
        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=LEARNING_RATE,
            weight_decay=WEIGHT_DECAY,
            betas=(0.9, 0.999),
        )
        print(f"  Optimizer: AdamW (lr={LEARNING_RATE}, wd={WEIGHT_DECAY})")
    else:
        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=LEARNING_RATE,
            weight_decay=WEIGHT_DECAY,
        )
        print(f"  Optimizer: Adam (lr={LEARNING_RATE})")

    # Scheduler: CosineAnnealingWarmRestarts
    if SCHEDULER_TYPE == "cosine_warm":
        scheduler = CosineAnnealingWarmRestarts(
            optimizer,
            T_0=T_0,
            T_mult=T_MULT,
            eta_min=ETA_MIN,
        )
        print(f"  Scheduler: CosineWarmRestarts (T_0={T_0}, T_mult={T_MULT}, eta_min={ETA_MIN})")
    else:
        scheduler = ReduceLROnPlateau(
            optimizer,
            mode="max",
            factor=0.5,
            patience=5,
            min_lr=ETA_MIN,
        )
        print(f"  Scheduler: ReduceLROnPlateau")

    # ── 4. Resume dari Checkpoint ────────────────────────────
    start_epoch      = 0
    best_val_f1      = -1.0
    patience_counter = 0
    history          = {
        "train_loss": [], "train_acc": [], "train_f1": [],
        "val_loss":   [], "val_acc":   [], "val_f1":   [],
        "lr":         [],
    }

    if os.path.exists(path_last) and not force_retrain:
        print(f"[4/7] ✓ Checkpoint ditemukan → Resume dari {path_last}")
        ckpt             = torch.load(path_last, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ckpt["model_state"])
        optimizer.load_state_dict(ckpt["optim_state"])
        try:
            scheduler.load_state_dict(ckpt["sched_state"])
        except:
            print("  [WARNING] Cannot load scheduler state, continuing without resume")
        start_epoch      = ckpt["epoch"] + 1
        best_val_f1      = ckpt["best_val_f1"]
        patience_counter = ckpt["patience_counter"]
        history          = ckpt["history"]
        print(f"  Resume dari epoch {start_epoch}, best_val_f1={best_val_f1:.4f}")
    else:
        print("[4/7] Training dari awal.")

    # ── 5. Training Loop ─────────────────────────────────────
    print(f"[5/7] Training (max {MAX_EPOCHS} epoch, patience={PATIENCE})...")
    print(f"  {'Epoch':>5} | {'TrLoss':>7} | {'TrF1':>6} | {'VaLoss':>7} | {'VaF1':>6} | {'LR':>8} | Status")
    print(f"  {'─'*5} | {'─'*7} | {'─'*6} | {'─'*7} | {'─'*6} | {'─'*8} | {'─'*10}")

    t_start = time.time()

    for epoch in range(start_epoch, MAX_EPOCHS):
        tr_loss, tr_acc, tr_f1       = train_epoch(model, loaders["train"], criterion, optimizer)
        va_loss, va_acc, va_f1, _, _ = evaluate(model, loaders["val"], criterion)

        current_lr = optimizer.param_groups[0]["lr"]
        
        # Step scheduler
        if SCHEDULER_TYPE == "cosine_warm":
            # CosineWarmRestarts step setiap epoch
            scheduler.step(epoch + va_f1 / 100)  # Add small fraction for stability
        else:
            scheduler.step(va_f1)

        history["train_loss"].append(tr_loss)
        history["train_acc"].append(tr_acc)
        history["train_f1"].append(tr_f1)
        history["val_loss"].append(va_loss)
        history["val_acc"].append(va_acc)
        history["val_f1"].append(va_f1)
        history["lr"].append(current_lr)

        status = ""
        if va_f1 > best_val_f1 + 1e-4:   # threshold kecil cegah noise
            best_val_f1      = va_f1
            patience_counter = 0
            status           = "★ BEST"
            torch.save({
                "model_state"  : model.state_dict(),
                "val_f1"       : va_f1,
                "epoch"        : epoch,
                "backbone"     : backbone_name,
                "lstm_type"    : lstm_type,
                "bidirectional": bidirectional,
                "use_attention": use_attention,
                "norm_mean"    : norm_stats["mean"],
                "norm_std"     : norm_stats["std"],
                "history"      : history,
            }, path_best)
        else:
            patience_counter += 1

        print(f"  {epoch+1:>5} | {tr_loss:>7.4f} | {tr_f1:>6.4f} | "
              f"{va_loss:>7.4f} | {va_f1:>6.4f} | {current_lr:>8.2e} | {status}")

        # Simpan checkpoint LAST (untuk resume)
        torch.save({
            "model_state"     : model.state_dict(),
            "optim_state"     : optimizer.state_dict(),
            "sched_state"     : scheduler.state_dict() if hasattr(scheduler, 'state_dict') else {},
            "epoch"           : epoch,
            "best_val_f1"     : best_val_f1,
            "patience_counter": patience_counter,
            "history"         : history,
            "backbone"        : backbone_name,
            "lstm_type"       : lstm_type,
            "bidirectional"   : bidirectional,
            "use_attention"   : use_attention,
            "norm_mean"       : norm_stats["mean"],
            "norm_std"        : norm_stats["std"],
        }, path_last)

        with open(path_hist, "w") as f:
            json.dump({k: [float(v) for v in vals]
                       for k, vals in history.items()}, f, indent=2)

        if patience_counter >= PATIENCE:
            print(f"\n  → Early stopping epoch {epoch+1} (patience {PATIENCE} habis).")
            break

    elapsed = time.time() - t_start
    print(f"\n[6/7] Selesai dalam {elapsed/60:.1f} menit.")
    print(f"  Best Val F1 Macro : {best_val_f1:.4f}")

    # ── 6. Evaluasi Test Set ─────────────────────────────────
    print("[7/7] Evaluasi Test Set...")
    ckpt_best = torch.load(path_best, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt_best["model_state"])

    te_loss, te_acc, te_f1, te_preds, te_labels = evaluate(
        model, loaders["test"], criterion
    )
    te_precision    = precision_score(te_labels, te_preds, average="macro", zero_division=0)
    te_recall       = recall_score(te_labels, te_preds, average="macro", zero_division=0)
    te_f1_per_class = f1_score(te_labels, te_preds, average=None, zero_division=0)
    te_cm           = confusion_matrix(te_labels, te_preds)

    try:
        model.eval()
        all_probs = []
        with torch.no_grad():
            for feats, _ in loaders["test"]:
                feats = feats.to(DEVICE)
                logits, _ = model(feats)
                probs = torch.softmax(logits, dim=1)[:, 0].cpu().numpy()
                all_probs.extend(probs)
        labels_inverted = [1 - l for l in te_labels]
        te_roc_auc = roc_auc_score(labels_inverted, all_probs)
    except Exception as e:
        print(f"  [WARNING] ROC-AUC calculation failed: {e}")
        te_roc_auc = float("nan")

    # Confusion matrix breakdown
    true_drowsy           = te_cm[0, 0]
    drowsy_missed         = te_cm[0, 1]  # FATAL: drowsy → notdrowsy
    notdrowsy_false_alarm = te_cm[1, 0]
    true_notdrowsy        = te_cm[1, 1]
    drowsy_recall         = true_drowsy / max(true_drowsy + drowsy_missed, 1)

    print(f"\n  ─── HASIL TEST SET ───────────────────────────")
    print(f"  Accuracy          : {te_acc:.4f}")
    print(f"  F1 Macro          : {te_f1:.4f}")
    print(f"  Precision         : {te_precision:.4f}")
    print(f"  Recall            : {te_recall:.4f}")
    print(f"  ROC-AUC           : {te_roc_auc:.4f}")
    print(f"  F1 drowsy         : {te_f1_per_class[0]:.4f}")
    print(f"  F1 notdrowsy      : {te_f1_per_class[1]:.4f}")
    print(f"  Drowsy Recall     : {drowsy_recall:.4f} (safety metric)")
    print(f"  ─── CONFUSION MATRIX ───────────────────────")
    print(f"  TP drowsy (aman)  : {true_drowsy}")
    print(f"  FATAL miss        : {drowsy_missed}")
    print(f"  False alarm       : {notdrowsy_false_alarm}")
    print(f"  TN notdrowsy      : {true_notdrowsy}")

    result = {
        "model_id"              : model_id,
        "backbone"              : backbone_name,
        "lstm_type"             : lstm_type,
        "bidirectional"         : bidirectional,
        "use_attention"         : use_attention,
        "fc_activation"         : FC_ACTIVATION,
        "best_val_f1"           : float(best_val_f1),
        "test_acc"              : float(te_acc),
        "test_f1_macro"         : float(te_f1),
        "test_precision"        : float(te_precision),
        "test_recall"           : float(te_recall),
        "test_roc_auc"          : float(te_roc_auc),
        "f1_drowsy"             : float(te_f1_per_class[0]),
        "f1_notdrowsy"          : float(te_f1_per_class[1]),
        "drowsy_recall"         : float(drowsy_recall),
        "confusion_matrix"      : te_cm.tolist(),
        "true_drowsy"           : int(true_drowsy),
        "drowsy_missed"         : int(drowsy_missed),
        "notdrowsy_false_alarm" : int(notdrowsy_false_alarm),
        "true_notdrowsy"        : int(true_notdrowsy),
        "elapsed_min"           : round(elapsed / 60, 2),
        "best_epoch"            : int(ckpt_best["epoch"]) + 1,
        "total_epochs_run"      : epoch + 1,
        "config"                : {
            "use_aug": USE_AUG,
            "aug_prob_fd": AUG_PROB_FD,
            "aug_prob_gn": AUG_PROB_GN,
            "learning_rate": LEARNING_RATE,
            "weight_decay": WEIGHT_DECAY,
            "max_grad_norm": MAX_GRAD_NORM,
            "label_smoothing": LABEL_SMOOTHING,
        }
    }

    result_path = os.path.join(model_dir, f"{model_id}_test_result.json")
    with open(result_path, "w") as f:
        json.dump(result, f, indent=2)

    print(f"\n✓ Semua file tersimpan: {model_dir}")

    # ── 7. Bersihkan GPU Memory ──────────────────────────────
    del model, loaders, optimizer, scheduler, criterion
    gc.collect()
    torch.cuda.empty_cache()
    print(f"  GPU cleared → {torch.cuda.memory_allocated()/1e6:.0f} MB allocated\n")

    return result


print("✓ Fungsi run_training (SWIN_EXP_D config) terdefinisi.")


✓ Fungsi run_training (SWIN_EXP_D config) terdefinisi.


## 7. Eksekusi Training — SWIN Saja

Notebook ini difokuskan hanya untuk backbone SWIN agar evaluasi lebih praktis dan cepat.
Pada bagian ini kita melatih:

- SWIN + LSTM
- SWIN + BiLSTM

In [22]:
all_results = []  # Tampung hasil fokus SWIN saja

print("\n")
print("╔" + "═" * 70 + "╗")
print("║  TRAINING SWIN ONLY (EXP_D Safety-Optimized Configuration)           ║")
print("║  Fokus: SWIN + LSTM/BiLSTM | Backbone lain dinonaktifkan             ║")
print("╚" + "═" * 70 + "╝")

print("\n[1/2] SWIN + LSTM (EXP_D CONFIG ✓)")
print("  └─ GELU activation, AdamW, CosineWarmRestarts")
print("  └─ Augmentation: Frame Drop 30% + Noise 40%")
print("  └─ LR=2e-4, WD=5e-4, MaxGradNorm=5.0, LabelSmoothing=0.05")
result_swin_lstm = run_training(
    backbone_name = "SWIN",
    bidirectional = False,
    use_attention = USE_ATTENTION,
    force_retrain = True,
)
all_results.append(result_swin_lstm)

print("\n[2/2] SWIN + BiLSTM (EXP_D CONFIG ✓)")
print("  └─ GELU activation, AdamW, CosineWarmRestarts")
print("  └─ Augmentation: Frame Drop 30% + Noise 40%")
result_swin_bilstm = run_training(
    backbone_name = "SWIN",
    bidirectional = True,
    use_attention = USE_ATTENTION,
    force_retrain = True,
)
all_results.append(result_swin_bilstm)

print("\n" + "╔" + "═" * 70 + "╗")
print("║  ✓ TRAINING SWIN SELESAI                                             ║")
print("╚" + "═" * 70 + "╝\n")



╔══════════════════════════════════════════════════════════════════════╗
║  TRAINING SWIN ONLY (EXP_D Safety-Optimized Configuration)           ║
║  Fokus: SWIN + LSTM/BiLSTM | Backbone lain dinonaktifkan             ║
╚══════════════════════════════════════════════════════════════════════╝

[1/2] SWIN + LSTM (EXP_D CONFIG ✓)
  └─ GELU activation, AdamW, CosineWarmRestarts
  └─ Augmentation: Frame Drop 30% + Noise 40%
  └─ LR=2e-4, WD=5e-4, MaxGradNorm=5.0, LabelSmoothing=0.05

  MODEL: SWIN_LSTM — SWIN_EXP_D Configuration
  Attention: True | Bidirectional: False
[1/7] Loading data & membangun DataLoader...


  [Normalisasi] feat_mean=0.2107, feat_std=0.2008
  [ClassWeight] drowsy=0.8944, notdrowsy=1.1339
  [Augmentation] ENABLED
[2/7] Membangun model...
  Total params: 1,415,042
[3/7] Building loss, optimizer, scheduler...
  Optimizer: AdamW (lr=0.0002, wd=0.0005)
  Scheduler: CosineWarmRestarts (T_0=10, T_mult=2, eta_min=1e-06)
[4/7] Training dari awal.
[5/7] Training (max 80 epoch, patience=15)...
  Epoch |  TrLoss |   TrF1 |  VaLoss |   VaF1 |       LR | Status
  ───── | ─────── | ────── | ─────── | ────── | ──────── | ──────────
      1 |  0.5560 | 0.6920 |  0.9944 | 0.5450 | 2.00e-04 | ★ BEST
      2 |  0.4070 | 0.8203 |  1.1860 | 0.5326 | 2.00e-04 | 
      3 |  0.3709 | 0.8481 |  1.3818 | 0.5087 | 1.95e-04 | 
      4 |  0.3397 | 0.8691 |  1.3965 | 0.5065 | 1.81e-04 | 
      5 |  0.3250 | 0.8794 |  1.3836 | 0.5239 | 1.59e-04 | 
      6 |  0.3046 | 0.8905 |  1.4232 | 0.5318 | 1.31e-04 | 
      7 |  0.2886 | 0.9016 |  1.4310 | 0.5281 | 1.00e-04 | 
      8 |  0.2719 | 0.9137 |  1.5147 | 

## 8. Ringkasan Hasil & Visualisasi SWIN

Menampilkan ringkasan metrik SWIN (LSTM vs BiLSTM) dan menyimpan kurva training.

In [23]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

# ─── Ranking Summary SWIN ─────────────────────────────────────
print("\n" + "=" * 80)
print("  RINGKASAN HASIL — SWIN_LSTM_EXP_D")
print("=" * 80)

if not all_results:
    print("Tidak ada hasil. Jalankan cell training terlebih dahulu.")
else:
    df_results = pd.DataFrame(all_results)

    # Sort by test_f1_macro
    df_sorted = df_results.sort_values("test_f1_macro", ascending=False).reset_index(drop=True)
    df_sorted.index = df_sorted.index + 1

    print("\nPeringkat berdasarkan F1 Macro")
    print("-" * 115)
    print(f"{'Rank':<4} {'Model ID':<30} {'F1_Macro':<10} {'F1_Drowsy':<10} {'Drowsy_Recall':<15} {'Missed_FN':<10} {'AUC':<8}")
    print("-" * 115)

    for rank, (_, row) in enumerate(df_sorted.iterrows(), 1):
        print(f"{rank:<4}  {str(row['model_id']):<30} {row['test_f1_macro']:<10.4f} "
              f"{row['f1_drowsy']:<10.4f} {row['drowsy_recall']:<15.4f} {int(row['drowsy_missed']):<10} "
              f"{row['test_roc_auc']:<8.4f}")

    print("\nPeringkat berdasarkan kriteria safety")
    print("-" * 115)
    df_safety = df_results[
        (df_results['drowsy_recall'] >= 0.45) &
        (df_results['drowsy_missed'] < 300) &
        (df_results['test_f1_macro'] >= 0.50)
    ].sort_values(["drowsy_recall", "drowsy_missed"], ascending=[False, True]).reset_index(drop=True)

    if df_safety.empty:
        print("Tidak ada model yang lolos semua safety criteria.")
        print("Kriteria: drowsy_recall >= 0.45, drowsy_missed < 300, F1_macro >= 0.50")
    else:
        df_safety.index = df_safety.index + 1
        for rank, (_, row) in enumerate(df_safety.iterrows(), 1):
            print(f"{rank:<4}  {str(row['model_id']):<30} {row['test_f1_macro']:<10.4f} "
                  f"{row['f1_drowsy']:<10.4f} {row['drowsy_recall']:<15.4f} {int(row['drowsy_missed']):<10} "
                  f"{row['test_roc_auc']:<8.4f}")

    print("\n" + "=" * 80)
    print("  BEST MODEL SUMMARY")
    print("=" * 80)

    best_academic = df_sorted.iloc[0]
    print(f"\nBest F1 Macro : {best_academic['model_id']}")
    print(f"F1 Macro      : {best_academic['test_f1_macro']:.4f}")
    print(f"Test Accuracy : {best_academic['test_acc']:.4f}")
    print(f"Drowsy Recall : {best_academic['drowsy_recall']:.4f}")
    print(f"Missed Drowsy : {int(best_academic['drowsy_missed'])}")
    print(f"ROC-AUC       : {best_academic['test_roc_auc']:.4f}")

    if not df_safety.empty:
        best_safety = df_safety.iloc[0]
        print(f"\nBest Safety   : {best_safety['model_id']}")
        print(f"F1 Macro      : {best_safety['test_f1_macro']:.4f}")
        print(f"Test Accuracy : {best_safety['test_acc']:.4f}")
        print(f"Drowsy Recall : {best_safety['drowsy_recall']:.4f}")
        print(f"Missed Drowsy : {int(best_safety['drowsy_missed'])}")
        print(f"ROC-AUC       : {best_safety['test_roc_auc']:.4f}")

    csv_path = os.path.join(SAVE_DIR, "SWIN_EXP_D_RESULTS.csv")
    df_results.to_csv(csv_path, index=False)
    print(f"\nHasil lengkap disimpan di: {csv_path}")

print("\n" + "=" * 80)

# ─── Plot training history untuk dua model SWIN ─────────────
def plot_training_history(model_id: str):
    """Plot loss, F1 Macro, dan LR dari history JSON."""
    model_dir = os.path.join(SAVE_DIR, model_id)
    hist_path = os.path.join(model_dir, f"{model_id}_history.json")

    if not os.path.exists(hist_path):
        print(f"[SKIP] History tidak ditemukan: {hist_path}")
        return

    with open(hist_path) as f:
        h = json.load(f)

    epochs = range(1, len(h["train_loss"]) + 1)
    best_ep = int(np.argmax(h["val_f1"])) + 1

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    fig.suptitle(f"Training History — {model_id} (SWIN_EXP_D)", fontsize=12, fontweight="bold")

    axes[0].plot(epochs, h["train_loss"], label="Train Loss", color="#2196F3", linewidth=1.5)
    axes[0].plot(epochs, h["val_loss"], label="Val Loss", color="#F44336", linewidth=1.5)
    axes[0].axvline(best_ep, color="green", linestyle="--", alpha=0.7, label=f"Best={best_ep}")
    axes[0].set_title("Loss")
    axes[0].legend()
    axes[0].set_xlabel("Epoch")
    axes[0].grid(alpha=0.3)

    axes[1].plot(epochs, h["train_f1"], label="Train F1", color="#2196F3", linewidth=1.5)
    axes[1].plot(epochs, h["val_f1"], label="Val F1", color="#F44336", linewidth=1.5)
    axes[1].axvline(best_ep, color="green", linestyle="--", alpha=0.7)
    axes[1].set_title("F1 Macro")
    axes[1].legend()
    axes[1].set_xlabel("Epoch")
    axes[1].grid(alpha=0.3)

    axes[2].plot(epochs, h["lr"], color="#9C27B0", linewidth=1.5)
    axes[2].axvline(best_ep, color="green", linestyle="--", alpha=0.7)
    axes[2].set_title("Learning Rate")
    axes[2].set_xlabel("Epoch")
    axes[2].grid(alpha=0.3)
    axes[2].yaxis.set_major_formatter(mtick.FormatStrFormatter("%.2e"))

    plt.tight_layout()
    save_path = os.path.join(model_dir, f"{model_id}_training_curve.png")
    plt.savefig(save_path, dpi=120, bbox_inches="tight")
    plt.close()
    print(f"Kurva tersimpan: {save_path}")


print("\nMembuat kurva training SWIN...")
for result in all_results:
    plot_training_history(result["model_id"])

print("Selesai.\n")


  RINGKASAN HASIL — SWIN_LSTM_EXP_D

Peringkat berdasarkan F1 Macro
-------------------------------------------------------------------------------------------------------------------
Rank Model ID                       F1_Macro   F1_Drowsy  Drowsy_Recall   Missed_FN  AUC     
-------------------------------------------------------------------------------------------------------------------
1     SWIN_BILSTM                    0.5697     0.5320     0.5766          232        0.5455  
2     SWIN_LSTM                      0.4884     0.3551     0.3120          377        0.5474  

Peringkat berdasarkan kriteria safety
-------------------------------------------------------------------------------------------------------------------
1     SWIN_BILSTM                    0.5697     0.5320     0.5766          232        0.5455  

  BEST MODEL SUMMARY

Best F1 Macro : SWIN_BILSTM
F1 Macro      : 0.5697
Test Accuracy : 0.5730
Drowsy Recall : 0.5766
Missed Drowsy : 232
ROC-AUC       : 0.5455

B

Kurva tersimpan: C:\kuliah-sementara\SKRIPSI\lstm_models_nthuddd2\SWIN_LSTM\SWIN_LSTM_training_curve.png
Kurva tersimpan: C:\kuliah-sementara\SKRIPSI\lstm_models_nthuddd2\SWIN_BILSTM\SWIN_BILSTM_training_curve.png
Selesai.

